# Librería

In [1]:
# Manipulacion de datos
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_columns', 200)
import json
import os
import unicodedata


from pathlib import Path

## Salida: `data/02_processed/datos_sequia.csv`

Una fila por **municipio × año agrícola × ciclo** (OI / PV), con el nivel de sequía del
Monitor de Sequía (D0-D4 → 0-4) agregado a **máximo** de las quincenas de cada mes:

| columna | ventana | qué mide |
|---|---|---|
| `nivel_sequia_max`        | meses del ciclo (OI: nov t-1 … abr t · PV: abr … sep t) | máx. de sequía **durante** el ciclo |
| `nivel_sequia_prev90_max` | 3 meses (~90 días) previos al inicio del ciclo (OI: ago-oct t-1 · PV: ene-mar t) | máx. de sequía **antecedente** (condición de arranque, previa al tratamiento) |

`NaN` = el municipio no aparece clasificado por el Monitor en ninguna quincena de la ventana.

In [2]:
df = pd.read_excel('/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_sequia/MunicipiosSequia.xlsx')

In [3]:
df

,CVE_CONCATENADA,CVE_ENT,CVE_MUN,NOMBRE_MUN,ENTIDAD,ORG_CUENCA*,CLV_OC,CON_CUENCA,CVE_CONC,2003-01-31 00:00:00,2003-02-28 00:00:00,2003-03-31 00:00:00,2003-04-30 00:00:00,2003-05-31 00:00:00,2003-06-30 00:00:00,2003-07-31 00:00:00,2003-08-31 00:00:00,2003-09-30 00:00:00,2003-10-31 00:00:00,2003-11-30 00:00:00,2003-12-31 00:00:00,2004-01-31 00:00:00,2004-02-29 00:00:00,2004-03-31 00:00:00,2004-04-30 00:00:00,2004-05-31 00:00:00,2004-06-30 00:00:00,2004-07-31 00:00:00,2004-08-31 00:00:00,2004-09-30 00:00:00,2004-10-31 00:00:00,2004-11-30 00:00:00,2004-12-31 00:00:00,2005-01-31 00:00:00,2005-02-28 00:00:00,2005-03-31 00:00:00,2005-04-30 00:00:00,2005-05-31 00:00:00,2005-06-30 00:00:00,2005-07-31 00:00:00,2005-08-31 00:00:00,2005-09-30 00:00:00,2005-10-31 00:00:00,2005-11-30 00:00:00,2005-12-31 00:00:00,2006-01-31 00:00:00,2006-02-28 00:00:00,2006-03-31 00:00:00,2006-04-30 00:00:00,2006-05-31 00:00:00,2006-06-30 00:00:00,2006-07-31 00:00:00,2006-08-31 00:00:00,2006-09-30 00:00:00,2006-10-31 00:00:00,2006-11-30 00:00:00,2006-12-31 00:00:00,2007-01-31 00:00:00,2007-02-28 00:00:00,2007-03-31 00:00:00,2007-04-30 00:00:00,2007-05-31 00:00:00,2007-06-30 00:00:00,2007-07-31 00:00:00,2007-08-31 00:00:00,2007-09-30 00:00:00,2007-10-31 00:00:00,2007-11-30 00:00:00,2007-12-31 00:00:00,2008-01-31 00:00:00,2008-02-29 00:00:00,2008-03-31 00:00:00,2008-04-30 00:00:00,2008-05-31 00:00:00,2008-06-30 00:00:00,2008-07-31 00:00:00,2008-08-31 00:00:00,2008-09-30 00:00:00,2008-10-31 00:00:00,2008-11-30 00:00:00,2008-12-31 00:00:00,2009-01-31 00:00:00,2009-02-28 00:00:00,2009-03-31 00:00:00,2009-04-30 00:00:00,2009-05-31 00:00:00,2009-06-30 00:00:00,2009-07-31 00:00:00,2009-08-31 00:00:00,2009-09-30 00:00:00,2009-10-31 00:00:00,2009-11-30 00:00:00,2009-12-31 00:00:00,2010-01-31 00:00:00,2010-02-28 00:00:00,2010-03-31 00:00:00,2010-04-30 00:00:00,2010-05-31 00:00:00,2010-06-30 00:00:00,2010-07-31 00:00:00,...,2022-06-15 00:00:00,2022-06-30 00:00:00,2022-07-15 00:00:00,2022-07-31 00:00:00,2022-08-15 00:00:00,2022-08-31 00:00:00,2022-09-15 00:00:00,2022-09-30 00:00:00,2022-10-15 00:00:00,2022-10-31 00:00:00,2022-11-15 00:00:00,2022-11-30 00:00:00,2022-12-15 00:00:00,2022-12-31 00:00:00,2023-01-15 00:00:00,2023-01-31 00:00:00,2023-02-15 00:00:00,2023-02-28 00:00:00,2023-03-15 00:00:00,2023-03-31 00:00:00,2023-04-30 00:00:00,2023-05-15 00:00:00,2023-05-31 00:00:00,2023-06-15 00:00:00,2023-06-30 00:00:00,2023-07-15 00:00:00,2023-07-31 00:00:00,2023-08-15 00:00:00,2023-08-31 00:00:00,2023-09-15 00:00:00,2023-09-30 00:00:00,2023-10-15 00:00:00,2023-10-31 00:00:00,2023-11-15 00:00:00,2023-11-30 00:00:00,2023-12-15 00:00:00,2023-12-31 00:00:00,2024-01-15 00:00:00,2024-01-31 00:00:00,2024-02-15 00:00:00,2024-02-29 00:00:00,2024-03-15 00:00:00,2024-03-31 00:00:00,2024-04-15 00:00:00,2024-04-30 00:00:00,2024-05-15 00:00:00,2024-05-31 00:00:00,2024-06-15 00:00:00,2024-06-30 00:00:00,2024-07-15 00:00:00,2024-07-31 00:00:00,2024-08-15 00:00:00,2024-08-31 00:00:00,2024-09-15 00:00:00,2024-09-30 00:00:00,2024-10-15 00:00:00,2024-10-31 00:00:00,2024-11-15 00:00:00,2024-11-30 00:00:00,2024-12-15 00:00:00,2024-12-31 00:00:00,2025-01-15 00:00:00,2025-01-31 00:00:00,2025-02-15 00:00:00,2025-02-28 00:00:00,2025-03-15 00:00:00,2025-03-31 00:00:00,2025-04-15 00:00:00,2025-04-30 00:00:00,2025-05-15 00:00:00,2025-05-31 00:00:00,2025-06-15 00:00:00,2025-06-30 00:00:00,2025-07-15 00:00:00,2025-07-31 00:00:00,2025-08-15 00:00:00,2025-08-31 00:00:00,2025-09-15 00:00:00,2025-09-30 00:00:00,2025-10-15 00:00:00,2025-10-31 00:00:00,2025-11-15 00:00:00,2025-11-30 00:00:00,2025-12-15 00:00:00,2025-12-31 00:00:00,2026-01-15 00:00:00,2026-01-31 00:00:00,2026-02-15 00:00:00,2026-02-28 00:00:00,2026-03-15 00:00:00,2026-03-31 00:00:00,2026-04-15 00:00:00,2026-04-30 00:00:00,2026-05-15 00:00:00,2026-05-31 00:00:00,2026-06-15 00:00:00,2026-06-30 00:00:00,2026-07-15 00:00:00,2026-07-31 00:00:00,2026-08-15 00:00:00
0,1001,1,1,Aguascalientes,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16

In [4]:
import pandas as pd

# 1. Identificar las columnas de fechas vs. metadatos dinámicamente
columnas_base = []
columnas_fecha_filtradas = []
fechas_dt_validas = [] # Guardaremos las fechas reales para usarlas después

for col in df.columns:
    # Convertimos el nombre de la columna a datetime de forma segura
    fecha_dt = pd.to_datetime(col, errors='coerce')
    
    # Si pd.to_datetime falló, es un metadato (como 'CVE_ENT' o 'ORG_CUENCA*')
    if pd.isna(fecha_dt):
        columnas_base.append(col)
    else:
        # Si es una fecha válida, verificamos si está en el rango 2013-2024
        if 2013 <= fecha_dt.year <= 2025:
            columnas_fecha_filtradas.append(col)
            fechas_dt_validas.append(fecha_dt)

# 2. Separar los metadatos y las fechas en dos DataFrames distintos
df_metadatos = df[columnas_base].copy()
df_fechas = df[columnas_fecha_filtradas].copy()

# 3. Reemplazar los valores categóricos por numéricos
mapeo_sequia = {'D1': 1, 'D2': 2, 'D3': 3, 'D4': 4, 'D0': 0}
df_fechas = df_fechas.replace(mapeo_sequia)

# 4. Cambiar el nombre de las columnas de fecha al formato YYYYMM
# Usamos la lista de fechas válidas que guardamos en el paso 1
df_fechas.columns = [fecha_dt.strftime('%Y%m') for fecha_dt in fechas_dt_validas]

# 5. Agrupar las quincenas y extraer el máximo mensual
df_fechas_max = df_fechas.T.groupby(level=0).max().T

# 6. Unir nuevamente los metadatos con las fechas consolidadas
df_final = pd.concat([df_metadatos, df_fechas_max], axis=1)

# Mostrar el resultado final
display(df_final.head())

/var/folders/zl/8w22zx5911s93bs5f33hkp040000gn/T/ipykernel_3004/1793600309.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_fechas = df_fechas.replace(mapeo_sequia)


,CVE_CONCATENADA,CVE_ENT,CVE_MUN,NOMBRE_MUN,ENTIDAD,ORG_CUENCA*,CLV_OC,CON_CUENCA,CVE_CONC,201301,201302,201303,201304,201305,201306,201307,201308,201309,201310,201311,201312,201401,201402,201403,201404,201405,201406,201407,201408,201409,201410,201411,201412,201501,201502,201503,201504,201505,201506,201507,201508,201509,201510,201511,201512,201601,201602,201603,201604,201605,201606,201607,201608,201609,201610,201611,201612,201701,201702,201703,201704,201705,201706,201707,201708,201709,201710,201711,201712,201801,201802,201803,201804,201805,201806,201807,201808,201809,201810,201811,201812,201901,201902,201903,201904,201905,201906,201907,201908,201909,201910,201911,201912,202001,202002,202003,202004,202005,202006,202007,202008,202009,202010,202011,202012,202101,202102,202103,202104,202105,202106,202107,202108,202109,202110,202111,202112,202201,202202,202203,202204,202205,202206,202207,202208,202209,202210,202211,202212,202301,202302,202303,202304,202305,202306,202307,202308,202309,202310,202311,202312,202401,202402,202403,202404,202405,202406,202407,202408,202409,202410,202411,202412,202501,202502,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512
0,1001,1,1,Aguascalientes,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,2.0,1.0,1.0,1.0,0.0,NaN,NaN,0.0,0.0,1.0,1.0,2.0,2.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0,1.0,3.0,3.0,2.0,2.0,2.0,3.0,2.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,1002,1,2,Asientos,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,2.0,2.0,1.0,0.0,NaN,NaN,NaN,1.0,1.0,2.0,2.0,2.0,2.0,3.0,3.0,3.0,3.0,3.0,1.0,2.0,2.0,0.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1003,1,3,Calvillo,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,2.0,1.0,1.0,2.0,1.0,0.0,NaN,0.0,0.0,0.0,1.0,1.0,2.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,1.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1004,1,4,Cosío,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,2.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,0.0,0.0,2.0,2.0,2.0,3.0,2.0,2.0,3.0,3.0,3.0,2.0,3.0,2.0,1.0,2.0,2.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [5]:
# Asumimos que 'df_final' y 'columnas_base' ya existen por el código anterior
anios = range(2013, 2025)
resultados = []

for anio in anios:
    # 1. Meses (YYYYMM) de la ventana de cada ciclo agrícola
    meses_oi = [f"{anio-1}11", f"{anio-1}12", f"{anio}01", f"{anio}02", f"{anio}03", f"{anio}04"]
    meses_pv = [f"{anio}04", f"{anio}05", f"{anio}06", f"{anio}07", f"{anio}08", f"{anio}09"]

    # 1b. Meses de los ~90 días (3 meses calendario) PREVIOS al inicio de cada ciclo
    #     OI arranca en nov de t-1  ->  ago, sep, oct de t-1
    #     PV arranca en abr de t    ->  ene, feb, mar de t
    meses_oi_prev = [f"{anio-1}08", f"{anio-1}09", f"{anio-1}10"]
    meses_pv_prev = [f"{anio}01", f"{anio}02", f"{anio}03"]

    # 2. Quedarnos solo con los meses que existen en los datos
    _val = lambda ms: [m for m in ms if m in df_final.columns]
    meses_oi_validos,      meses_pv_validos      = _val(meses_oi),      _val(meses_pv)
    meses_oi_prev_validos, meses_pv_prev_validos = _val(meses_oi_prev), _val(meses_pv_prev)

    # 3. DataFrame temporal para el año actual
    df_anio = df_final[columnas_base].copy()
    df_anio['anio'] = anio

    # 4. Máximo de sequía DURANTE la ventana del ciclo
    df_anio['OI'] = df_final[meses_oi_validos].max(axis=1) if meses_oi_validos else np.nan
    df_anio['PV'] = df_final[meses_pv_validos].max(axis=1) if meses_pv_validos else np.nan

    # 4b. Máximo de sequía en los ~90 días PREVIOS al inicio del ciclo
    df_anio['OI__prev'] = df_final[meses_oi_prev_validos].max(axis=1) if meses_oi_prev_validos else np.nan
    df_anio['PV__prev'] = df_final[meses_pv_prev_validos].max(axis=1) if meses_pv_prev_validos else np.nan

    resultados.append(df_anio)

# Unir todos los años en un DataFrame temporal
df_temporal = pd.concat(resultados, ignore_index=True)

# 5. Pasar a formato largo: una fila por municipio x anio x ciclo
df_ciclos = df_temporal.melt(
    id_vars=columnas_base + ['anio'],
    value_vars=['OI', 'PV'],
    var_name='nomcicloproductivo',   # "OI" u "PV"
    value_name='nivel_sequia_max',   # max. de sequia DURANTE el ciclo
)

# 5b. La metrica previa: mismo melt (mismo orden OI->PV) -> se alinea fila a fila
_prev = df_temporal.melt(
    id_vars=columnas_base + ['anio'],
    value_vars=['OI__prev', 'PV__prev'],
    var_name='nomcicloproductivo',
    value_name='nivel_sequia_prev90_max',   # max. de sequia en los ~90 dias previos al inicio del ciclo
)
assert (_prev['nomcicloproductivo'].str.replace('__prev', '', regex=False).values
        == df_ciclos['nomcicloproductivo'].values).all(), 'los dos melt quedaron desalineados'
df_ciclos['nivel_sequia_prev90_max'] = _prev['nivel_sequia_prev90_max'].values

# (Opcional) nombres expandidos:
# mapeo_ciclos = {'OI': 'Otoño-Invierno', 'PV': 'Primavera-Verano'}
# df_ciclos['nomcicloproductivo'] = df_ciclos['nomcicloproductivo'].replace(mapeo_ciclos)

display(df_ciclos.head(10))

,CVE_CONCATENADA,CVE_ENT,CVE_MUN,NOMBRE_MUN,ENTIDAD,ORG_CUENCA*,CLV_OC,CON_CUENCA,CVE_CONC,anio,nomcicloproductivo,nivel_sequia_max,nivel_sequia_prev90_max
0,1001,1,1,Aguascalientes,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
1,1002,1,2,Asientos,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
2,1003,1,3,Calvillo,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
3,1004,1,4,Cosío,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
4,1005,1,5,Jesús María,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
5,1006,1,6,Pabellón de Arteaga,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
6,1007,1,7,Rincón de Romos,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
7,1008,1,8,San José de Gracia,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
8,1009,1,9,Tepezalá,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN
9,1010,1,10,El Llano,Aguascalientes,Lerma-Santiago-Pacífico,VIII,Rio Santiago,16,2013,OI,NaN,NaN


In [6]:
print('max. de sequia DURANTE el ciclo (nivel_sequia_max):')
display(df_ciclos.groupby('nivel_sequia_max', dropna=False).size().reset_index(name='count'))

print('max. de sequia en los ~90 dias PREVIOS al ciclo (nivel_sequia_prev90_max):')
display(df_ciclos.groupby('nivel_sequia_prev90_max', dropna=False).size().reset_index(name='count'))

max. de sequia DURANTE el ciclo (nivel_sequia_max):


,nivel_sequia_max,count
0,0.0,20472
1,1.0,14312
2,2.0,7600
3,3.0,3619
4,4.0,775
5,NaN,12694


max. de sequia en los ~90 dias PREVIOS al ciclo (nivel_sequia_prev90_max):


,nivel_sequia_prev90_max,count
0,0.0,19253
1,1.0,10332
2,2.0,4636
3,3.0,2095
4,4.0,320
5,NaN,22836


In [7]:
df_ciclos.to_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_sequia.csv', index=False)